- Fazer a Friction Ellipse para ambos os casos
- Fazer max, min, e mean velocidade
- Fazer o RMSE do predicted com o real

Vão ser precisos dois bags, com e sem a NN excepto no RMSE do predicted com o Real

Nível a seguir é fazer intervalos de segurança

In [ ]:
import os
import shutil
import yaml
import bagpy
from bagpy import bagreader
import pandas as pd
import numpy as np
from scipy.integrate import solve_ivp
from tkinter import Tk     # from tkinter import Tk for Python 3.x
from tkinter.filedialog import askopenfilename
from MPCModelFuncs import differential_mpc_model
from MPCModelFuncs import euler_step
import matplotlib.pyplot as plt
import scipy.io as sio

In [ ]:
# Select Bags
path_bag_normal = '/home/fst/david_bags/google_20/2023-07-04-16-48-50.bag'
path_bag_NN = '/home/fst/david_bags/google_20/trained/2023-07-04-15-27-23.bag'

bag_normal = bagreader(path_bag_normal)
bag_NN = bagreader(path_bag_NN)

NORMAL_MSG = bag_normal.message_by_topic('/control/learning_mpcc/nn_data')
NN_MSG = bag_NN.message_by_topic('/control/learning_mpcc/nn_data')

normal_data = pd.read_csv(NORMAL_MSG)
nn_data = pd.read_csv(NN_MSG)

In [ ]:
vel_min = 3.0
vel_normal = np.sqrt(normal_data['car_velocity.velocity.x']**2 + normal_data['car_velocity.velocity.y']**2)
vel_NN = np.sqrt(nn_data['car_velocity.velocity.x']**2 + nn_data['car_velocity.velocity.y']**2)

vx_normal = normal_data['car_velocity.velocity.x'][vel_normal>vel_min]
vy_normal = normal_data['car_velocity.velocity.y'][vel_normal>vel_min]
yaw_rate_normal = normal_data['car_velocity.velocity.theta'][vel_normal>vel_min]
accel_x_normal = normal_data['car_acceleration.x'][vel_normal>vel_min]
accel_y_normal = normal_data['car_acceleration.y'][vel_normal>vel_min]

vx_NN = nn_data['car_velocity.velocity.x'][vel_NN>vel_min]
vy_NN = nn_data['car_velocity.velocity.y'][vel_NN>vel_min]
yaw_rate_NN = nn_data['car_velocity.velocity.theta'][vel_NN>vel_min]
accel_x_NN = nn_data['car_acceleration.x'][vel_NN>vel_min]
accel_y_NN = nn_data['car_acceleration.y'][vel_NN>vel_min]

progress_normal = normal_data['progress']
progress_NN = nn_data['progress']

In [ ]:
# Plot friction ellipses
plt.scatter(accel_x_normal, accel_y_normal, c="blue", label="Normal")
plt.scatter(accel_x_NN, accel_y_NN, c="red", label="NN")
plt.title("Friction Ellipses")
plt.xlabel("Longitudinal Acceleration [m/s^2]")
plt.ylabel("Lateral Acceleration [m/s^2]")
plt.legend()
plt.show()


In [ ]:
# Stability Envelope
PI = 3.14159265358979323846
Beta_normal = np.arctan2(vy_normal, vx_normal)/PI*360
Beta_NN = np.arctan2(vy_NN, vx_NN)/PI*360

plt.scatter(Beta_normal, yaw_rate_normal/PI*360, c="blue", label="Normal", s=0.7)
plt.scatter(Beta_NN, yaw_rate_NN/PI*360, c="red", label="NN", s=0.7)

plt.title("Stability Envelope")
plt.xlabel("Beta [deg]")
plt.ylabel("Yaw Rate [deg/s]")

plt.legend()

plt.show()

In [ ]:
# Velocity per time
plt.plot(range(len(vx_normal)), np.sqrt(vx_normal**2 + vy_normal**2), c="blue", label="Normal",linewidth=0.5)
plt.plot(range(len(vx_NN)), np.sqrt(vx_NN**2 +  vy_NN**2), c="red", label="NN",linewidth=0.5)
plt.legend()
plt.title("Velocity per time")
plt.savefig("../saved_images/velocity_time_profile.eps", dpi=1200)
plt.show()

In [ ]:
# Velocity Stats
vel_normal_min = np.min(vel_normal)
vel_normal_max = np.max(vel_normal)

vel_NN_min = np.min(vel_NN)
vel_NN_max = np.max(vel_NN)

vel_normal_mean = np.mean(vel_normal)
vel_NN_mean = np.mean(vel_NN)

print("Normal - max: {:.2f}, mean: {:.2f}".format(vel_normal_max, vel_normal_mean))
print("NN     - max: {:.2f}, mean: {:.2f}".format(vel_NN_max, vel_NN_mean))

In [ ]:
# Velocity per progress
progress_vel_normal = progress_normal[vel_normal>vel_min]
progress_vel_NN = progress_NN[vel_NN>vel_min]
plt.scatter(progress_vel_normal, np.sqrt(vx_normal**2 + vy_normal**2), c="blue", label="Normal", s=0.7)
plt.scatter(progress_vel_NN, np.sqrt(vx_NN**2 +  vy_NN**2), c="red", label="NN", s=0.7)
plt.legend()
plt.title("Velocity per progress")
plt.savefig("../saved_images/velocity_progress_profile.eps", dpi=1200)
plt.show()

In [ ]:
# Compare Progresses
threshold = 6.0
idx_normal = next((i for i, x in enumerate(progress_normal) if x >= threshold), len(progress_normal))
comp_progress_normal = progress_normal[idx_normal:]
idx_NN = next((i for i, x in enumerate(progress_NN) if x >= threshold), len(progress_NN))
comp_progress_NN = progress_NN[idx_NN:]
plt.plot(range(len(comp_progress_normal)), comp_progress_normal, c="blue", label="Normal",linewidth=0.5)
plt.plot(range(len(comp_progress_NN)), comp_progress_NN, c="red", label="NN",linewidth=0.5)
plt.legend()
plt.title("Progress")
plt.savefig("../saved_images/progress_profile.eps", dpi=1200)
plt.show()

In [ ]:
print(len(normal_data['car_velocity.velocity.x']))
print(len(np.sqrt(normal_data['car_velocity.velocity.x']**2 + normal_data['car_velocity.velocity.y']**2)))
type(normal_data['car_velocity.velocity.x'])

In [ ]:
# Select Bags
path_bag_pp= '/home/fst/david_bags/google_20/2023-07-04-18-26-59.bag'

bag_pp= bagreader(path_bag_pp)


PP_MSG = bag_pp.message_by_topic('/control/controller/nn_data')

pp_data = pd.read_csv(PP_MSG)

vel_min = 3.0
vel_pp = np.sqrt(pp_data['car_velocity.velocity.x']**2 + pp_data['car_velocity.velocity.y']**2)

progress_vel_pp = pp_data['progress'][vel_pp>vel_min].to_numpy()
vel_pp = vel_pp[vel_pp>vel_min].to_numpy()

In [ ]:
vel_normal = np.sqrt(vx_normal**2 + vy_normal**2)
vel_NN = np.sqrt(vx_NN**2 +  vy_NN**2)
progress_vel_normal = progress_vel_normal.to_numpy()
progress_vel_NN = progress_vel_NN.to_numpy()
vel_normal = vel_normal.to_numpy()
vel_NN = vel_NN.to_numpy()

data = {
    'progress_vel_normal' : progress_vel_normal,
    'progress_vel_NN' : progress_vel_NN,
    'progress_vel_pp' : progress_vel_pp,
    'vel_normal' : vel_normal,
    'vel_NN' : vel_NN,
    'vel_pp' : vel_pp
}
# print(data['progress_vel_normal'])
# print(data['progress_vel_NN'])
sio.savemat('metrics.mat', data)